# EHR IHD QML walkthrough

**RESEARCH/EDUCATIONAL EXPERIMENT - NOT A MEDICAL DIAGNOSIS**

This notebook separates heart-disease presence classification from ten-year incident coronary heart-disease risk prediction. It calls reusable project modules and reuses a protected completed smoke run when available; on a clean workspace it executes the bounded UCI smoke configuration.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from qml_research.ehr_ihd.config import load_ihd_config
from qml_research.ehr_ihd.data import dataset_report, deterministic_split, load_uci_cleveland
from qml_research.ehr_ihd.predict import predict_record
from qml_research.ehr_ihd.runner import run_uci_smoke

config = load_ihd_config(Path("../experiments/ehr_ihd_qml/configs/uci_smoke.yaml"))
dataset = load_uci_cleveland()
split = deterministic_split(dataset, config.seed)
dataset_report(dataset, split)

## Provenance, label, and leakage prevention

UCI `num == 0` is absence and `num in 1..4` is presence. The outcome is never an input. Imputation, scaling, and consensus selection fit on training rows only. Validation selects calibration and threshold; test data is used only for final metrics. Framingham instead excludes prevalent CHD and requires adequate follow-up for a ten-year negative.

In [ ]:
runs_root = Path("../results/ehr_ihd_qml")
completed = sorted(runs_root.glob("uci-smoke-*/summary.json")) if runs_root.exists() else []
if completed:
    run_dir = completed[-1].parent
    summary = json.loads(completed[-1].read_text(encoding="utf-8"))
else:
    summary = run_uci_smoke(config)
    run_dir = runs_root / summary["run_id"]
summary

## Feature selection, baselines, OQSVM, and HQMLP

Training-only logistic RFE, random-forest importance, and mutual information produce a deterministic consensus. The OQSVM uses a two-repeat Pauli Z/ZZ statevector fidelity kernel. The preferred HQMLP uses BCE logits; the paper-inspired U3/CNOT ablation uses MSE. Classical and reduced quantum metrics are reported together only when their patient/feature scope is explicit.

In [ ]:
selected = json.loads((run_dir / "selected_features.json").read_text(encoding="utf-8"))
rankings = json.loads((run_dir / "feature_rankings.json").read_text(encoding="utf-8"))
metrics = pd.read_csv(run_dir / "metrics/model_metrics.csv")
display(selected)
display(
    metrics[
        ["model", "auroc", "auprc", "brier_score", "selected_threshold", "training_time_seconds"]
    ]
)

## Calibration, explanations, runtime, and circuit/kernel counts

Raw margins or quantum outputs are not probabilities. Platt calibration and the decision threshold are fitted on validation data. Bootstrap intervals quantify small-test uncertainty. Permutation importance and feature ablation describe model behaviour, not clinical causality.

In [ ]:
kernel_cost = json.loads((run_dir / "scaling/kernel_cost.json").read_text(encoding="utf-8"))
model_metrics = json.loads((run_dir / "metrics/model_metrics.json").read_text(encoding="utf-8"))
display(kernel_cost)
display(
    [
        {k: row.get(k) for k in ["model", "confidence_intervals", "circuit_kernel_evaluations"]}
        for row in model_metrics
    ]
)

## Held-out patient prediction

The interface rejects outcomes and invalid ranges, restores the exact feature order, emits a model score and calibrated probability, and applies the validation-selected threshold.

In [ ]:
record = {
    "age": 58,
    "sex": 1,
    "cp": 2,
    "trestbps": 140,
    "chol": 240,
    "fbs": 0,
    "restecg": 1,
    "thalach": 150,
    "exang": 0,
    "oldpeak": 1.2,
    "slope": 1,
    "ca": 0,
    "thal": 2,
}
prediction = predict_record(run_dir, record, explain=True)
display(prediction)

## Comparison with the paper and quantum-advantage conclusion

The source paper used a private Basurto EHR cohort, ten selected features, a Pauli statevector QSVM, and a hybrid QMLP. This public experiment improves split, leakage, calibration, AUPRC, and cost reporting but cannot directly reproduce the private cohort. A small statevector smoke run cannot establish clinical validity or computational quantum advantage. Predictive quantum utility would require stable paired gains over strong feature-matched baselines across seeds with uncertainty; hardware utility and asymptotic advantage require separate evidence. Negative findings are valid.

The Framingham early-risk section remains awaiting the verified official teaching file. No cohort or metric is fabricated.